In [ ]:
from bddl.knowledge_base import *

In [ ]:
# Get the substance synsets that are missing a system
for s in Synset.all_objects():
    if s.is_substance and len(s.matching_particle_systems) == 0:
        print(s)

In [ ]:
# Get the particle systems that are missing a particle
missing_particle = ParticleSystem.view_error_missing_particle()
print(len(missing_particle))

In [ ]:
# Remove the particle systems that are cooked
missing_particle_not_cooked = [
    ps for ps in missing_particle if "cooked__" not in ps.name
]
print(len(missing_particle_not_cooked))
print(missing_particle_not_cooked)

In [ ]:
# Remove the particle systems that are not diced
missing_particle_not_cooked_not_diced = [
    ps for ps in missing_particle_not_cooked if "diced__" not in ps.name
]
print(len(missing_particle_not_cooked_not_diced))
print(missing_particle_not_cooked_not_diced)

In [ ]:
# Count by group
import collections

ctr = collections.Counter(
    [("cooked__" in ps.name, "diced__" in ps.name) for ps in missing_particle]
)
print("Uncooked Diced:", ctr[(False, True)])
print("Cooked Diced:", ctr[(True, True)])
print("Uncooked Non-Diced:", ctr[(False, False)])
print("Cooked Non-Diced:", ctr[(True, False)])

In [ ]:
# How many non-liquid particle systems are there anyway?
print(
    "Total non-liquid particle systems:",
    sum([1 for s in Synset.all_objects() if s.is_substance and not s.is_liquid]),
)

In [ ]:
# And how many diced?
print(
    "Total diced particle systems:",
    sum([1 for s in Synset.all_objects() if s.is_substance and "diced__" in s.name]),
)
print(
    "Diced particle systems w/ particles:",
    sum(
        [
            1
            for s in Synset.all_objects()
            if s.is_substance and "diced__" in s.name and s.state == SynsetState.MATCHED
        ]
    ),
)

In [ ]:
# Get a list of all cookable substances
for s in Synset.all_objects():
    if s.is_substance and "cookable" in s.property_names:
        print(s)

In [ ]:
# For all existing particle systems, get the average max extent of their particles
import numpy as np

systems_and_sizes = []
for ps in ParticleSystem.all_objects():
    if not any("macro" in pn for pn in ps.synset.property_names):
        continue
    if len(ps.particles) > 0:
        avg = np.mean([np.max(p.bounding_box_size) for p in ps.particles])
        systems_and_sizes.append((ps, avg))

# Sort by size
systems_and_sizes.sort(key=lambda x: x[1])
for ps, size in systems_and_sizes:
    print(ps, size)

In [ ]:
# Diced-not-cooked particle system record for Wensi's pass
import json

diced_not_cooked = []
for ps in missing_particle_not_cooked:
    if "diced__" in ps.name and "cooked__" not in ps.name:
        # Get the definition of the derivative parent
        parents = [
            da
            for da in ps.synset.derivative_ancestors
            if not da.is_derivative and "sliced__" not in da.name
        ]
        definition = None
        if len(parents) > 0:
            assert len(parents) == 1, f"Expected 1 parent for {ps.name}, got {parents}"
            parent = parents[0]
            definition = parent.definition
        diced_not_cooked.append((ps.name, definition))
with open("diced_particle_systems.json", "w") as f:
    json.dump(diced_not_cooked, f, indent=4)